### 💡 Train / Validation / Test

모델이 학습 데이터의 정답을 잘 맞힌다고 해서, 처음보는 데이터도 잘 예측한다는 보장은 없다.

이 문제를 해결하기 위해 데이터를 **Train(훈련)**, **Validation(검증)**, **Test(테스트)** 세 가지로 나눈다.

주어진 데이터를 훈련, 검증, 테스트로 나누어야 하는데 그 이유가 무엇일까

예를들어 학생 100명의 공부 시간과 시험 점수가 있다고 했을 때, 모델이 이 데이터 100개를 전부 학습하고

같은 100개의 데이터로 성능을 평가하게 된다면 모델이 학습했던 데이터를 이미 모두 알고있기 때문에 평가 결과가 실제보다 더 좋게 나올 수 있다.

✅ 때문에 데이터를 아래와 같이 분류한다.

- `Train` : 모델 학습
    모델이 입력 X와 정답 y의 관계를 학습하는 데이터
    ```python
    # Train 데이터는 모델의 가중치나 계수를 학습하는데 사용된다.
    model.fit(X_train, y_train)
    ```

- `Validation` : 모델 선택 및 조정
    여러 모델이나 하이퍼파라미터를 비교하는 데이터
    ```python
    # Validation 데이터는 모델이 처음보는 데이터에 얼마나 잘 대응하는지 확인하면서,
    # 어떤 모델을 사용할지 결정하는데 도움을 준다.
    y_val_pred = model.predict(X_val)
    ```

- `Test` : 최종 성능 평가
    모델 선택을 끝낸 뒤 새로운 데이터에 대한 성능을 평가하는 데이터
    ```python
    # Test 데이터는 모델과 설정을 결정한 뒤 최종 평가에 사용된다.
    y_test_pred = model.predict(X_test)
    ```

In [15]:
# scikit-learn에서 제공하는 유방암 진단 데이터셋(Breast Cancer Wisconsin)을 사용해서 데이터 분류 해보기
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

cancer = load_breast_cancer(as_frame=True)

X = cancer.data
y = cancer.target

# X : 전체 Feature 대상
# y : 전체 Target 대상
# test_size=0.2 : 테스트 데이터의 비율을 20%로 설정
# random_state=42 : 무작위 분할 결과를 재현 가능하게 고정
# stratify=y : 각 클래스의 비율을 최대한 유지하며 분할
#               예를들어 전체 데이터가 양성 90%, 악성 10%라면 Test에서도 비슷한 비율을 유지하도록 설정
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 지금 Train 8 / Test 2의 비율로 분리되었으니
# Train에서 25%를 분리해서 Validation을 만들면
# Train 6 / Validation 2 / Test 2 가 만들어진다.
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    random_state=42,
    test_size=0.25,
    stratify=y_train
)

# Train 데이터의 평균과 표준편차를 계산
# Train 데이터를 표준화
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

# LogisticRegression 모델 학습
model.fit(X_train, y_train)

# Validation 데이터로 성능 확인하기
# Accuracy(정확도)는 전체 예측 중 맞힌 비율이다.
y_val_pred = model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
print("성능 확인 :", val_accuracy)

# Test 데이터로 최종 평가하기
y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print("최종 평가 :", test_accuracy)


성능 확인 : 0.9912280701754386
최종 평가 : 0.9824561403508771


위에 코드에서 사용한 `make_pipeline()`은 전처리 순서를 안전하게 관리하는데 도움을 준다.

예를들어 스케일링 후 데이터를 `Train`, `Validation`, `Test` 로 분리를 한다면

`Validation`, `Test의` 데이터 통계 정보가 `Train` 전처리에 반영이 되는 것이다.

이러한 상황을 **데이터 누수**(Data Leakage)라고 한다.

올바른 순서는 다음과 같다.

> 데이터 분할을 먼저 수행 &rightarrow; Train 데이터로만 scaler학습 &rightarrow; Validation과 Test에는 transform()만 수행

---
### 실습문제 

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

cancer = load_breast_cancer(as_frame=True)

X = cancer.data
y = cancer.target

# 1. 전체 데이터의 X.shape, y.shape을 출력해 봐.
print(X.shape)
print(y.shape)

# 2. train_test_split()을 사용해 Train 80%, Test 20%로 나눠 봐. random_state=42, stratify=y를 적용해.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. 분리한 Train 데이터에서 25%를 Validation으로 다시 나눠 봐. random_state=42, stratify=y_train을 적용해.
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

# 4. 최종 X_train, X_val, X_test의 shape을 각각 출력해 봐.
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

# 5. 왜 두 번째 분할에서 test_size=0.25를 사용하는지 설명해 봐.
# train, validation, test의 비율을 6/2/2로 나누기 위해서 기존 8의 비율을 가진 train에서 0.25의 비율을 나누면 6/2/2가 맞춰지기 때문

# 1. StandardScaler와 LogisticRegression을 make_pipeline()으로 연결하기
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

# 2. Train 데이터로 모델 학습하기
model.fit(X_train, y_train)

# 3. Validation 데이터로 예측하기
y_val_pred = model.predict(X_val)

# 4. accuracy_score()로 Validation 정확도 계산하기
val_accuracy = accuracy_score(y_val, y_val_pred)

# 5. Test 데이터로 예측하고 정확도 계산하기
y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)

# 6. Validation과 Test 데이터를 구분해야 하는 이유 설명하기
# Validation은 학습된 데이터를 검증하는 구간이고 해당 구간에서 여러 모델 선택 등을 한다. 그리고 Test에서는 마지막 최종 테스트이다.

(569, 30)
(569,)
(341, 30)
(114, 30)
(114, 30)
